# Comparing DEA Methods (DESeq2 vs edgeR-like)



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from rnaseq_toolkit.dea import run_deseq2, run_edger_like

# Load data
counts = pd.read_csv('../data/example_counts.csv', index_col=0)


In [ ]:
res_deseq2 = run_deseq2(counts, meta, design='~condition', contrast=['condition', 'Treated', 'Control'])
sig_deseq2 = res_deseq2[(res_deseq2['padj'] < 0.05) & (res_deseq2['log2FoldChange'].abs() > 1)].dropna()


In [ ]:
res_edger = run_edger_like(counts, meta, group_col='condition', contrast=('Treated', 'Control'))
sig_edger = res_edger[(res_edger['padj'] < 0.05) & (res_edger['log2FoldChange'].abs() > 1)].dropna()


In [ ]:
common_genes = res_deseq2.index.intersection(res_edger.index)
lfc_d = res_deseq2.loc[common_genes, 'log2FoldChange'].dropna()
lfc_e = res_edger.loc[lfc_d.index, 'log2FoldChange'].dropna()

shared = lfc_d.index.intersection(lfc_e.index)
r_val, _ = pearsonr(lfc_d[shared], lfc_e[shared])
print(f"Log2FC Pearson correlation: r = {r_val:.3f}")

plt.figure(figsize=(6, 6))
plt.scatter(lfc_d[shared], lfc_e[shared], alpha=0.5, s=10)
plt.xlabel('DESeq2 Log2FC')
plt.ylabel('edgeR-like Log2FC')
plt.title(f'Method Concordance (r={r_val:.3f})')
plt.axline((0, 0), slope=1, color='red', linestyle='--')
